**使用第三方天气查询API，实现完整可执行的天气查询应用**
包含：函数调用 + 智谱 glm-4 工具调用 + 真实天气接口免费天气 API（Open-Meteo）

In [16]:
import json
import requests
import os
from tenacity import retry, stop_after_attempt, wait_random_exponential
#from tenacity import retry 作用：自动重试失败的请求
#stop_after_attempt 作用：设置最多重试几次
#wait_random_exponential 作用：重试时等待一会儿，越来越久（退避指数）

from dotenv import load_dotenv
from termcolor import colored
# 自动加载 .env 文件里的所有 key
load_dotenv()

True

In [29]:
# 定义一个函数pretty_print_conversation，用于打印消息对话内容
def pretty_print_conversation(messages):

    # 为不同角色设置不同的颜色
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "tools": "magenta",
    }

    # 遍历消息列表
    for message in messages:

        # 如果消息的角色是"system"，则用红色打印“content”
        if message["role"] == "system":
            print(colored(f"system: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"user"，则用绿色打印“content”
        elif message["role"] == "user":
            print(colored(f"user: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"assistant"，并且消息中包含"tool_calls"，则用蓝色打印"⏳ 正在查询中，请稍候..."
        elif message["role"] == "assistant" and message.get("tool_calls"):
            print(colored(f"assistant[tool_calls]:'⏳ 正在查询中，请稍候...'\n", role_to_color[message["role"]]))

        # 如果消息的角色是"assistant"，但是消息中不包含"tool_choice"，则用蓝色打印“content”
        elif message["role"] == "assistant" and not message.get("tool_calls"):
            print(colored(f"assistant[content]: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"function"，则用品红色打印“function”
        elif message["role"] == "tools":
            print(colored(f"function ({message['name']}): {message['content']}\n", role_to_color[message["role"]]))


In [30]:
# 1. 真实天气查询函数（调用第三方天气API）
def get_weather(location):
    """
    调用第三方天气API，获取实时天气
    :param location: 城市名，如 "Shanghai"
    :return: 天气信息
    """
    try:
        # 第一步：根据城市名获取经纬度
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=zh&format=json"
        geo_resp = requests.get(geo_url).json()

        if not geo_resp.get("results"):
            return {"error": "找不到该城市"}

        lat = geo_resp["results"][0]["latitude"]
        lon = geo_resp["results"][0]["longitude"]
        city = geo_resp["results"][0]["name"]
        
        # 第二步：获取天气
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather = requests.get(weather_url).json()

        return {
            "城市": city,
            "温度": weather["current_weather"]["temperature"],
            "风速": weather["current_weather"]["windspeed"],
            "天气代码": weather["current_weather"]["weathercode"]
        }

    except Exception as e:
        return {"error": str(e)}


In [31]:
# 2. 智谱 API 请求函数
# ==============================================
@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, tools=None, tool_choice="auto", model="glm-4.5-air"):
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + os.getenv("ZHIPUAI_API_KEY"),
    }

    json_data = {
        "model": model,
        "messages": messages
    }

    # 如果传入了tools，将其加入到json_data中
    if tools is not None:
        json_data.update({"tools": tools})

    # tool_choice，将其加入到json_data中
    if tool_choice is None:
        json_data["tool_choice"] = "auto"
    else:
        json_data["tool_choice"] = tool_choice

    # 尝试发送POST请求到OpenAI服务器的chat/completions接口
    try:
        response = requests.post(
            "https://open.bigmodel.cn/api/paas/v4/chat/completions",
            headers=headers,
            json=json_data,
        )
        # 返回服务器的响应
        return response

    # 如果发送请求或处理响应时出现异常，打印异常信息并返回
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e


In [32]:
# 3. 工具定义（给大模型调用）
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取当前天气。用户问天气必须调用，禁止直接回答！",  # 功能的描述
        "parameters": {  # 定义该功能需要的参数
            "type": "object",
            "properties": {  # 参数的属性
                "location": {  # 地点参数
                    "type": "string",  # 参数类型为字符串
                    "description": "城市名，例如 上海",  # 参数的描述
                }
            },
            "required": ["location"],  # 该功能需要的必要参数
            }
        }
    }
]

In [33]:
# 4. 执行工具调用
def execute_function_call(message):
    tool_call = message["tool_calls"][0]
    func_name = tool_call["function"]["name"]
    args = json.loads(tool_call["function"]["arguments"])

    if func_name == "get_weather":
        return json.dumps(get_weather(args["location"]), ensure_ascii=False)
    else:
        return json.dumps({"error": f"函数 {func_name} 不存在"})

In [34]:
# 5. 完整对话流程
messages = [
    {"role": "system", "content": "你是天气助手，必须调用工具查询天气，不能编造答案"},
    {"role": "user", "content": "上海今天天气怎么样？"}
]

# 第一次调用：模型决定调用天气工具
chat_response = chat_completion_request(messages, tools=tools)
resp_json = chat_response.json()
assistant_message = resp_json["choices"][0]["message"]
messages.append(assistant_message)

# 执行工具
if assistant_message.get("tool_calls"):
    tool_result = execute_function_call(assistant_message)
    tool = assistant_message["tool_calls"][0]

    # 把结果返回给模型
    messages.append({
        "role": "tool",
        "tool_call_id": tool["id"],
        "name": tool["function"]["name"],
        "content": tool_result
    })

    # 第二次调用：模型生成自然语言回答
    final_resp = chat_completion_request(messages, tools=tools)
    final_msg = final_resp.json()["choices"][0]["message"]
    messages.append(final_msg)
    pretty_print_conversation(messages)

system: 你是天气助手，必须调用工具查询天气，不能编造答案

user: 上海今天天气怎么样？

assistant[tool_calls]:'⏳ 正在查询中，请稍候...'

assistant[content]: 
根据查询结果，上海今天的天气情况如下：

🌤️ **上海今日天气**
- **温度**：22.6°C
- **风速**：13.2 km/h  
- **天气状况**：晴天（天气代码0）

今天上海天气不错，温度适宜，有轻微的风，是个晴朗的好天气！适合外出活动。



模型收到用户问题：“上海今天天气怎么样？”
第一次调用：只能生成 tool_calls，无法同时生成回答
工具执行：必须在你的本地代码里完成，模型没法直接访问天气 API
第二次调用：模型拿到工具结果，才能生成回答